# EDA — Tweets financiers + FiQA-SA

Objectif (RQ2) : mesurer l'effet d'un **changement de domaine dans la même langue** (anglais).
FPB = phrases de dépêches ; ici = tweets et microblogs/titres.

Comme pour FPB, chaque section répond à une question qui mène à une décision
pour `prepare_tweets.py` et `prepare_fiqa.py`.

## 0. Imports et chemins

In [ ]:
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Le notebook tourne depuis notebooks/ : on remonte à la racine du projet
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
TWEETS_DIR = ROOT / "data" / "raw" / "twitter_financial_news"
FIQA_DIR = ROOT / "data" / "raw" / "fiqa_sa"

LABELS = ["negative", "neutral", "positive"]
# Couleurs par polarité (comme le notebook 01) : rouge = négatif, gris = neutre, bleu = positif
COLORS = {"negative": "#e34948", "neutral": "#8c8b86", "positive": "#2a78d6"}
# Couleurs par jeu de données (identité, ordre fixe)
DATASET_COLORS = {"FPB": "#2a78d6", "Tweets": "#eb6834", "FiQA": "#1baf7a"}

pd.set_option("display.max_colwidth", 200)

# Référence "news" : le train de FPB déjà préparé (jamais le test)
fpb = pd.read_parquet(ROOT / "data" / "processed" / "fpb.parquet")
fpb_train = fpb[fpb["split"] == "train"]
len(fpb_train)

### Deux outils réutilisés dans tout le notebook

- `class_shares` : répartition des classes (en %) de plusieurs jeux ;
- `plot_shares` : la même chose en barres empilées (même graphique que dans le notebook 01).

In [ ]:
def class_shares(groups: dict[str, pd.Series]) -> pd.DataFrame:
    """{nom: série de labels} -> tableau effectifs + pourcentages par classe."""
    counts = (pd.DataFrame({name: s.value_counts() for name, s in groups.items()}).T
              .reindex(columns=LABELS).fillna(0).astype(int))
    counts["total"] = counts.sum(axis=1)
    pct = counts[LABELS].div(counts["total"], axis=0).mul(100).round(1).add_suffix(" %")
    return pd.concat([counts, pct], axis=1)


def plot_shares(table: pd.DataFrame, title: str) -> None:
    """Barres horizontales empilées : une barre par jeu, un segment par classe."""
    fig, ax = plt.subplots(figsize=(8, 0.7 * len(table) + 1.5))
    left = pd.Series(0.0, index=table.index)
    for label in LABELS:
        share = table[f"{label} %"]
        ax.barh(table.index, share, left=left, color=COLORS[label], label=label,
                edgecolor="white", linewidth=2)
        for y, (l, s) in enumerate(zip(left, share)):
            if s >= 4:  # pas d'étiquette sur les segments trop fins
                ax.text(l + s / 2, y, f"{s:.0f} %", ha="center", va="center",
                        color="white", fontsize=9)
        left += share
    ax.invert_yaxis()
    ax.set_xlabel("% des exemples")
    ax.set_title(title)
    ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

---
# Partie A — Twitter Financial News

## A1. Charger les fichiers

→ Question : quels splits existent ? (FPB n'en avait aucun, on les a créés nous-mêmes.)

In [ ]:
tweets = {s: pd.read_csv(TWEETS_DIR / f"sent_{s}.csv") for s in ["train", "valid"]}
for s, d in tweets.items():
    print(f"{s:5s} : {len(d):5d} lignes, colonnes = {list(d.columns)}")
tweets["train"].head()

## A2. Que veulent dire les labels 0, 1, 2 ?

Le README annonce `0 = Bearish, 1 = Bullish, 2 = Neutral`.
Notre schéma est `0 = negative, 1 = neutral, 2 = positive` : **1 et 2 sont inversés**.

On ne croit pas le README sur parole : on **lit** des exemples de chaque chiffre
et on vérifie que le sens correspond.

In [ ]:
t = tweets["train"]
for raw_id in sorted(t["label"].unique()):
    print(f"\n===== label brut {raw_id} =====")
    for text in t[t["label"] == raw_id].sample(6, random_state=0)["text"]:
        print("-", text)

Si la lecture confirme le README, on traduit **par nom** (Bearish → negative…),
jamais en recopiant les chiffres.
On garde le chiffre d'origine dans `raw_label` pour pouvoir revérifier.

In [ ]:
TWEET_LABELS = {0: "negative", 1: "positive", 2: "neutral"}  # Bearish, Bullish, Neutral

tweets = {s: d.rename(columns={"label": "raw_label"})
               .assign(label=lambda x: x["raw_label"].map(TWEET_LABELS))
          for s, d in tweets.items()}
tweets_all = pd.concat([d.assign(split=s) for s, d in tweets.items()], ignore_index=True)
pd.crosstab(tweets_all["raw_label"], tweets_all["label"])  # chaque chiffre -> un seul nom

## A3. Répartition des classes, comparée à FPB

→ Décision : stratifier le split, garder la macro-F1, et savoir si le déséquilibre
est différent de celui de FPB (un modèle entraîné sur FPB « a appris » les proportions de FPB).

In [ ]:
shares = class_shares({
    "FPB (train)": fpb_train["label"],
    "Tweets train": tweets["train"]["label"],
    "Tweets valid": tweets["valid"]["label"],
})
shares

In [ ]:
plot_shares(shares, "Répartition des classes : FPB vs tweets")

## A4. Doublons, conflits, fuite entre train et valid

Particularité des tweets : la **même dépêche** est souvent tweetée plusieurs fois avec un
lien raccourci différent (`https://t.co/...`). Deux tweets identiques à l'URL près sont des
doublons « cachés ». On compare donc aussi les textes **sans URL**.

In [ ]:
URL = r"https?://\S+"

def without_urls(text: str) -> str:
    return " ".join(re.sub(URL, "", text).split())

rows = []
for s, d in tweets.items():
    for version, texts in [("texte brut", d["text"]), ("sans URL", d["text"].map(without_urls))]:
        df = pd.DataFrame({"text": texts, "label": d["label"]})
        uniq = df.drop_duplicates()
        rows.append({"split": s, "version": version, "lignes": len(df),
                     "doublons exacts": df.duplicated().sum(),
                     "textes en conflit": uniq["text"].duplicated(keep=False).sum()})
pd.DataFrame(rows).set_index(["split", "version"])

In [ ]:
# Fuite : un même texte (sans URL) présent en train ET en valid ?
train_texts = set(tweets["train"]["text"].map(without_urls))
valid_texts = set(tweets["valid"]["text"].map(without_urls))
print(f"{len(train_texts & valid_texts)} textes communs à train et valid (sans URL)")

# Tweets qui ne contiennent QU'UNE URL : vides une fois l'URL retirée
empty = tweets_all[tweets_all["text"].map(without_urls) == ""]
print(f"{len(empty)} tweets vides sans leur URL :", empty["label"].value_counts().to_dict())

In [ ]:
# Exemples de doublons cachés (identiques une fois l'URL retirée)
d = tweets_all.assign(clean=tweets_all["text"].map(without_urls))
d[d["clean"].duplicated(keep=False)].sort_values("clean")[["text", "label", "split"]].head(10)

### Décision : option B (comme pour FPB)

Le découpage d'origine a une petite fuite (textes communs à train et valid) et n'a pas de test.
On **fusionne** train + valid, on **nettoie** (clé = texte sans URL), et seulement **ensuite**
on fera notre propre split stratifié 70/15/15 dans `prepare_tweets.py`.
Une phrase ne peut alors plus se retrouver des deux côtés.

Simulation du nettoyage pour voir combien de tweets il reste :

In [ ]:
merged = tweets_all.assign(clean=tweets_all["text"].map(without_urls))
not_empty = merged[merged["clean"] != ""]                          # retire les tweets réduits à une URL
no_dup = not_empty.drop_duplicates(subset=["clean", "label"])      # même texte (URL ignorée), même label
conflict = no_dup["clean"].duplicated(keep=False)                  # même texte, labels différents
clean_tweets = no_dup[~conflict]

print(f"fusion train + valid : {len(merged)}")
print(f"  - vides sans URL   : {len(merged) - len(not_empty)}")
print(f"  - doublons         : {len(not_empty) - len(no_dup)}")
print(f"  - en conflit       : {conflict.sum()}")
print(f"= tweets gardés      : {len(clean_tweets)}")
class_shares({"Tweets nettoyés": clean_tweets["label"]})

## A5. Ce qui n'existe pas dans FPB : URL, cashtags, mentions, emojis

→ Décision : que faire de chaque élément dans `prepare_tweets.py`
(le garder, le supprimer, ou le remplacer par un jeton comme `URL`).

In [ ]:
PATTERNS = {
    "URL": URL,
    "cashtag ($TSLA)": r"\$[A-Za-z][A-Za-z.]*",
    "mention (@user)": r"@\w+",
    "hashtag (#)": r"#\w+",
    "non ASCII (emoji…)": r"[^\x00-\x7F]",
}
t = tweets["train"]
pd.DataFrame({
    name: {"% tweets": t["text"].str.contains(p).mean() * 100,
           "% phrases FPB": fpb_train["text"].str.contains(p).mean() * 100}
    for name, p in PATTERNS.items()
}).T.round(1)

In [ ]:
for name, p in PATTERNS.items():
    print(f"\n===== {name} =====")
    for text in t[t["text"].str.contains(p)].sample(3, random_state=0)["text"]:
        print("-", text)

Question piège : la **présence d'une URL** est-elle liée au label ?
Si oui, un modèle pourrait apprendre « URL → neutral » au lieu du sens de la phrase :
c'est un raccourci qui ne marchera pas sur d'autres données.

In [ ]:
has_url = t["text"].str.contains(URL).map({True: "avec URL", False: "sans URL"})
pd.crosstab(has_url, t["label"], normalize="index").mul(100).round(1)[LABELS]

---
# Partie B — FiQA-SA

## B1. Charger les fichiers

FiQA vient d'une tâche **par aspect** : une phrase, une *cible* (entreprise), un *aspect*,
et un **score continu** de −1 à +1 (pas 3 classes).

In [ ]:
fiqa = {s: pd.read_parquet(FIQA_DIR / f"{s}.parquet") for s in ["train", "valid", "test"]}
fiqa_all = pd.concat([d.assign(split=s) for s, d in fiqa.items()], ignore_index=True)
for s, d in fiqa.items():
    print(f"{s:5s} : {len(d):4d} lignes")
fiqa_all.head()

In [ ]:
# Deux types de textes : titres de presse (headline) et microblogs (post)
fiqa_all.groupby(["split", "type"]).size().unstack()

## B2. Distribution du score

→ Question : le score est-il concentré autour de 0 (beaucoup de « neutre »)
ou plutôt tranché ?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(-1, 1, 41)
for typ, color in [("headline", "#2a78d6"), ("post", "#eb6834")]:
    ax.hist(fiqa_all.loc[fiqa_all["type"] == typ, "score"], bins=bins, histtype="step",
            linewidth=2, color=color, label=typ)
ax.axvline(0, color="#52514e", linestyle="--", linewidth=1)
ax.set_xlabel("score de sentiment (−1 = très négatif, +1 = très positif)")
ax.set_ylabel("nombre de lignes")
ax.set_title("FiQA : distribution du score par type de texte")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

fiqa_all.groupby("type")["score"].describe().round(3)

## B3. Passer du score à 3 classes : quel seuil ?

Règle : `score < −s` → negative, `score > s` → positive, sinon neutral.
Le choix de `s` change complètement la taille de la classe « neutral ».

→ Décision : un seuil **justifié** (et identique pour train/valid/test).

In [ ]:
def discretize(scores: pd.Series, s: float) -> pd.Series:
    return pd.Series(np.select([scores < -s, scores > s], ["negative", "positive"], "neutral"),
                     index=scores.index)

THRESHOLDS = [0.0, 0.05, 0.1, 0.2, 0.3]
by_threshold = class_shares({f"seuil ±{s}": discretize(fiqa_all["score"], s) for s in THRESHOLDS})
by_threshold

In [ ]:
plot_shares(by_threshold, "FiQA : répartition des classes selon le seuil")

Lire les phrases **proches de la frontière** : à partir de quel score une phrase
cesse-t-elle d'être neutre pour un investisseur ? (consigne FPB)

In [ ]:
near = fiqa_all[fiqa_all["score"].abs() <= 0.25].sort_values("score")
near[["score", "sentence"]].iloc[::max(1, len(near) // 20)]  # ~20 phrases réparties

## B4. Une phrase, plusieurs cibles

Comme FiQA note le sentiment **envers une cible**, une même phrase peut apparaître
plusieurs fois avec des scores différents.
Pour nous (une phrase → un label), ce sont des doublons ou des conflits.

In [ ]:
multi = fiqa_all[fiqa_all["sentence"].duplicated(keep=False)].sort_values("sentence")
print(f"{multi['sentence'].nunique()} phrases apparaissent plusieurs fois ({len(multi)} lignes)")
multi[["sentence", "target", "aspect", "score", "split"]].head(12)

In [ ]:
# Après discrétisation (ex. seuil 0.1) : combien de ces phrases ont des labels différents ?
# Et une même phrase est-elle répartie sur plusieurs splits (fuite) ?
g = fiqa_all.assign(label=discretize(fiqa_all["score"], 0.1)).groupby("sentence")
print("phrases en conflit de label    :", (g["label"].nunique() > 1).sum())
print("phrases sur plusieurs splits   :", (g["split"].nunique() > 1).sum())

## B5. Les mêmes éléments « réseaux sociaux » que dans les tweets ?

Les `post` sont des microblogs (StockTwits) : on réutilise les motifs de A5.

In [ ]:
pd.DataFrame({
    name: fiqa_all.groupby("type")["sentence"].apply(lambda s: s.str.contains(p).mean() * 100)
    for name, p in PATTERNS.items()
}).round(1)

---
# Partie C — Les trois domaines côte à côte

## C1. Longueur des textes (en mots)

→ Décision (étape 5) : `max_length` des Transformers pour les trois jeux.

In [ ]:
texts = {
    "FPB": fpb_train["text"],
    "Tweets": tweets["train"]["text"].map(without_urls),
    "FiQA": fiqa_all["sentence"].map(without_urls),
}
lengths = pd.DataFrame({name: s.str.split().str.len().describe(percentiles=[0.5, 0.95, 0.99])
                        for name, s in texts.items()}).T.round(1)
lengths

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = range(0, 60, 2)
for name, s in texts.items():
    n_words = s.str.split().str.len()
    ax.hist(n_words, bins=bins, histtype="step", linewidth=2, density=True,
            color=DATASET_COLORS[name], label=name)
ax.set_xlabel("nombre de mots")
ax.set_ylabel("proportion des textes")
ax.set_title("Longueur des textes par domaine")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## C2. Vocabulaire : à quel point les tweets et FiQA sont-ils « nouveaux » pour un modèle FPB ?

Taux de mots **hors vocabulaire** : la part des mots d'un jeu jamais vus dans le train de FPB.
C'est une mesure directe du changement de domaine pour la baseline TF-IDF :
un mot jamais vu à l'entraînement n'a aucun poids dans le modèle.

In [ ]:
def words(text: str) -> list[str]:
    return re.findall(r"[a-z]+", text.lower())

fpb_vocab = {w for text in fpb_train["text"] for w in words(text)}

rows = {}
for name, s in texts.items():
    tokens = [w for text in s for w in words(text)]
    unseen = [w for w in tokens if w not in fpb_vocab]
    rows[name] = {"mots (occurrences)": len(tokens),
                  "% hors vocabulaire FPB": round(len(unseen) / len(tokens) * 100, 1),
                  "mots inconnus les plus fréquents": ", ".join(
                      w for w, _ in Counter(unseen).most_common(12))}
pd.DataFrame(rows).T

## C3. Répartition des classes des trois jeux

(FiQA avec un seuil provisoire de 0.1, à confirmer en conclusion.)

In [ ]:
plot_shares(class_shares({
    "FPB (train)": fpb_train["label"],
    "Tweets (train)": tweets["train"]["label"],
    "FiQA (seuil 0.1)": discretize(fiqa_all["score"], 0.1),
}), "Répartition des classes par domaine")

---
## D. Conclusions → décisions pour `prepare_tweets.py` et `prepare_fiqa.py`

À remplir ensemble après avoir lu les résultats :

**Tweets**
- Correspondance des labels : 0 = Bearish → negative, 1 = Bullish → positive, 2 = Neutral → neutral,
  traduite **par nom** (vérifiée en lisant les exemples, A2).
- Split : **option B**. Fusion train + valid → nettoyage → split stratifié 70/15/15, seed 42 (comme FPB).
  Raisons : fuite de 32 textes entre train et valid dans le découpage d'origine, et pas de test.
- Doublons cachés : dédoublonnage et conflits calculés sur le **texte sans URL**, **avant** le split ;
  les tweets réduits à une URL sont retirés.
- URL / cashtags / mentions : …
- Caractères mal encodés (`â€¦` au lieu de `…`) : …

**FiQA**
- Seuil de discrétisation : …
- Phrases à plusieurs cibles : …
- Splits (garder ceux d'origine ?) : …

**Comparaison**
- Changement de domaine attendu (hors vocabulaire, longueur, classes) : …
- `max_length` envisagé : …